# 👕 Clothed Try-On — REAL AI backend (CatVTON, FREE GPU)

Run this notebook on **Kaggle** (recommended: free GPU T4, 30 hrs/week, no card) or **Google Colab** (free tier). It loads the REAL CatVTON try-on model and gives you a **public URL** that our website calls.

**Kaggle setup (2 min, one-time):**
1. https://www.kaggle.com → Sign up → verify your **phone number** (free, needed for GPU).
2. **Create → New Notebook** → upload/run this notebook (or paste cells).
3. Right-side **Settings** panel: **Accelerator = GPU (T4)**, **Internet = ON** (both required!).
4. **Run all cells** top to bottom. The last cell prints a `https://....gradio.live` URL.
5. Paste that URL into the website's `BACKEND_URL` (see main README) → real try-ons!
6. Keep this tab open while using the site. When done, **stop the session** to save your weekly GPU hours.

**Colab setup:** Runtime → Change runtime type → **T4 GPU** → run all cells. Same URL step. (Idle timeouts are stricter than Kaggle.)

In [ ]:
# Cell 2 — Install small extra packages.
# (torch with GPU support is PRE-INSTALLED on Kaggle/Colab — never reinstall it.)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers", "transformers", "accelerate", "peft",
                "gradio", "huggingface_hub", "opencv-python-headless",
                "torchvision"], check=True)
print("packages OK")

In [ ]:
# Cell 3 — Fetch the official CatVTON source code (tiny ~3 MB download).
import os, subprocess, sys
SRC_DIR = "catvton_src"
if not os.path.exists(os.path.join(SRC_DIR, "model", "pipeline.py")):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://huggingface.co/spaces/zhengchong/CatVTON", SRC_DIR],
                   check=True)
sys.path.insert(0, SRC_DIR)
print("official CatVTON code ready")

In [ ]:
# Cell 4 — Load the REAL CatVTON mask-free model onto the free GPU.
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), ("No GPU! Kaggle: right-panel Settings -> Accelerator -> GPU. "
                                   "Colab: Runtime -> Change runtime type -> T4 GPU. Then re-run.")

from huggingface_hub import snapshot_download
from utils import init_weight_dtype
from model.pipeline import CatVTONPix2PixPipeline

print("Downloading weights (~4 GB, one-time per session)...")
repo_path = snapshot_download(repo_id="zhengchong/CatVTON-MaskFree")
pipe = CatVTONPix2PixPipeline(
    base_ckpt="timbrooks/instruct-pix2pix",
    attn_ckpt=repo_path,
    attn_ckpt_version="mix-48k-1024",
    weight_dtype=init_weight_dtype("fp16"),  # fp16 = fits free T4 GPUs (bf16 needs A100-class)
    use_tf32=True,
    device="cuda",
)
print("REAL CatVTON engine loaded on GPU!")

In [ ]:
# Cell 5 — Start the public server. COPY THE gradio.live URL IT PRINTS.
# (On a T4, each try-on takes ~1-3 min. Steps=30 is the fast default; raise to 50 for finer detail.)
import gradio as gr
from PIL import Image
from utils import resize_and_crop, resize_and_padding

WIDTH, HEIGHT = 576, 768  # smaller = fits free 16 GB T4 memory (still sharp HD result)

def tryon(person_path, garment_path, seed=42, steps=30):
    """Person + garment file paths in, REAL AI result image out."""
    if not person_path or not garment_path:
        raise gr.Error("Upload BOTH a person photo and a garment photo.")
    person = Image.open(person_path).convert("RGB")
    garment = Image.open(garment_path).convert("RGB")
    person_r = resize_and_crop(person, (WIDTH, HEIGHT))
    garment_r = resize_and_padding(garment, (WIDTH, HEIGHT))
    generator = None
    if int(seed) != -1:
        generator = torch.Generator(device="cuda").manual_seed(int(seed))
    return pipe(image=person_r, condition_image=garment_r,
                num_inference_steps=int(steps), guidance_scale=2.5, height=HEIGHT, width=WIDTH,
                generator=generator)[0]

with gr.Blocks(title="Clothed Try-On (REAL CatVTON)") as demo:
    gr.Markdown("## 👕 Clothed Try-On — REAL AI\nPerson photo + garment photo → you wearing it. Keep this notebook running while you use the website.")
    with gr.Row():
        person_in = gr.Image(label="Your photo (full/upper body)", type="filepath")
        garment_in = gr.Image(label="Garment photo", type="filepath")
    with gr.Row():
        seed_in = gr.Slider(minimum=-1, maximum=10000, step=1, value=42, label="Seed (-1 = random)")
        steps_in = gr.Slider(minimum=10, maximum=100, step=5, value=30, label="Steps (30 = fast, 50 = finer)")
    btn = gr.Button("✨ Try it on", variant="primary")
    out = gr.Image(label="Result (REAL AI — not an overlay!)")
    btn.click(tryon, [person_in, garment_in, seed_in, steps_in], out, api_name="tryon")

print("Starting public server — copy the https://....gradio.live URL below into the website's BACKEND_URL.")
demo.launch(share=True, show_error=True)

## ✅ Done! Now connect the website

1. Copy the **`https://xxxxxxxx.gradio.live`** URL from the cell above.
2. In `frontend/index.html`, set `BACKEND_URL` to that URL (keep `BACKEND_TYPE = "gradio"`, `DEMO_MODE = false`).
3. Open the site → upload photo → pick garment → **real AI result** in ~1–3 min.

**Troubleshooting:** `No GPU!` → enable the GPU accelerator (see top) and re-run. `share link fails on Kaggle` → Settings → **Internet = ON**, then re-run the last cell. `Out of quota` on Kaggle → weekly hours reset; Colab free tier as backup. When finished: stop/shutdown the session so you don't burn free hours.